# KCAU Debate Club Elections 2026/27
## Data Processing & Vote Counting Audit Notebook

---

**Purpose:** This notebook provides a transparent, step-by-step walkthrough of how the election data was processed, cleaned, validated, and counted. It serves as an official audit trail for the KCAU Debate Club Elections 2026/27.

**Audience:** Both technical and non-technical stakeholders — club officials, election committee members, and any interested party.

**Data Sources:**
- `KCAU DEBATE CLUB MEMBERS.xlsx` — Official member register
- `KCAU Debate Club Elections 2026_27 Ballot (Responses).xlsx` — Google Forms ballot responses

**Date of Processing:** July 2026

---

## Step 1: Import Required Libraries

We start by importing the Python libraries we'll use:
- **pandas** — for data manipulation and analysis
- **openpyxl** — to read Excel files
- **re** — for pattern matching (regex) to validate registration numbers

In [21]:
# Import necessary libraries
import pandas as pd
import re
import plotly.express as px
from collections import Counter

print("Libraries loaded successfully")
print(f"pandas version: {pd.__version__}")

Libraries loaded successfully
pandas version: 3.0.5


---

##  Step 2: Load the Raw Data

We load two Excel files:
1. **Member Register** — Contains the official list of registered debate club members
2. **Ballot Responses** — Contains all ballot submissions from the Google Forms election

Let's load them and take a first look at the data.

In [22]:
# Load the member register
members_df = pd.read_excel('KCAU DEBATE CLUB MEMBERS.xlsx')

print("MEMBER REGISTER")
print(f"   Total rows: {len(members_df)}")
print(f"   Columns: {list(members_df.columns)}")
print()
members_df.head(10)

MEMBER REGISTER
   Total rows: 41
   Columns: ['REG NO', 'MEMBERS']



,REG NO,MEMBERS
0,25/08756,Abdullahi Osman
1,22/05032,Abrose Mwangi
2,26/03018,Akankunda Shallon
3,25/05989,Amillia Rehan
4,25/08062,Billy Lbaruni
5,24/06576,Churchil John Wesley
6,22/08970,Dan Nameas
7,22/06872,David Wekesa
8,25/07987,Derrick Kipchumba
9,22/06569,Ezekiel


In [23]:
# Load the ballot responses
ballots_df = pd.read_excel('KCAU Debate Club Elections 2026_27 Ballot (Responses).xlsx')

print("BALLOT RESPONSES")
print(f"   Total rows: {len(ballots_df)}")
print(f"   Columns: {list(ballots_df.columns)}")
print()
ballots_df.head(10)

BALLOT RESPONSES
   Total rows: 39
   Columns: ['Timestamp', 'Score', 'REGISTRATION NUMBER', 'PRESIDENT', 'VICE PRESIDENT', 'SECRETARY GENERAL', 'ORGANISING SECRETARY', 'PUBLICITY SECRETARY', 'FINANCE SECRETARY']



,Timestamp,Score,REGISTRATION NUMBER,PRESIDENT,VICE PRESIDENT,SECRETARY GENERAL,ORGANISING SECRETARY,PUBLICITY SECRETARY,FINANCE SECRETARY
0,2026-07-23 19:14:43.017,0,22/07839,Newton Ombaka,Akankunda Shallon,Osman Abdullahi,Josephat Mulei,Nyamungu Achieng,Yolanda Nyambura Waceke
1,2026-07-23 19:15:16.599,0,26/04092,Newton Ombaka,Amilia Rehan,Osman Abdullahi,Fiona Bett,Nyamungu Achieng,Yolanda Nyambura Waceke
2,2026-07-23 19:16:03.209,0,25/03330,Newton Ombaka,Amilia Rehan,Osman Abdullahi,George Mwai,Nyamungu Achieng,Yolanda Nyambura Waceke
3,2026-07-23 19:16:13.391,0,25/07551,Newton Ombaka,Akankunda Shallon,Osman Abdullahi,George Mwai,Nyamungu Achieng,Yolanda Nyambura Waceke
4,2026-07-23 19:16:24.009,0,26/03730,Newton Ombaka,Akankunda Shallon,Osman Abdullahi,Fiona Bett,Nyamungu Achieng,Yolanda Nyambura Waceke
5,2026-07-23 19:16:41.504,0,23/08299,Newton Ombaka,Akankunda Shallon,Osman Abdullahi,George Mwai,Nyamungu Achieng,Yolanda Nyambura Waceke
6,2026-07-23 19:17:04.568,0,25/04368,Newton Ombaka,Amilia Rehan,Osman Abdullahi,George Mwai,Nyamungu Achieng,Yolanda Nyambura Waceke
7,2026-07-23 19:17:10.757,0,26/03673,Newton Ombaka,Amilia Rehan,Osman Abdullahi,Josephat Mulei,Nyamungu Achieng,Yolanda Nyambura Waceke
8,2026-07-23 19:18:06.652,0,25/08096,Newton Ombaka,Akankunda Shallon,Osman Abdullahi,George Mwai,Nyamungu Achieng,Yolanda Nyambura Waceke
9,2026-07-23 19:17:49.112,0,26/00038,Newton Ombaka,Amilia Rehan,Osman Abdullahi,Josephat Mulei,Nyamungu Achieng,Yolanda Nyambura Waceke


---

## Step 3: Data Cleaning

Before we can count votes, we need to clean the data. This involves:

1. **Trimming whitespace** from registration numbers (some may have trailing spaces)
2. **Standardizing text** — ensuring consistency in names and formatting
3. **Identifying duplicate registrations** in the member list
4. **Validating registration number format** — expected format is `YY/NNNNN` (e.g., `25/08756`)

### Why is this important?
Dirty data can lead to incorrect vote counts. For example, a registration number with a trailing space (`25/05953 `) would not match the same number without a space (`25/05953`), potentially causing a legitimate member's vote to be classified as a non-member vote.

In [24]:
# 3a. Clean registration numbers
# Trim whitespace from all registration numbers
members_df['REG NO'] = members_df['REG NO'].astype(str).str.strip()
members_df['MEMBERS'] = members_df['MEMBERS'].astype(str).str.strip()

ballots_df['REGISTRATION NUMBER'] = ballots_df['REGISTRATION NUMBER'].astype(str).str.strip()

print("Registration numbers cleaned (whitespace trimmed)")

Registration numbers cleaned (whitespace trimmed)


In [25]:
# 3b. Check for duplicate registrations in the member list
# If the same registration number appears more than once, we flag it.

duplicate_members = members_df[members_df.duplicated(subset='REG NO', keep=False)]

if len(duplicate_members) > 0:
    print(f"DUPLICATE REGISTRATIONS FOUND: {len(duplicate_members)} rows")
    print()
    print(duplicate_members[['REG NO', 'MEMBERS']].to_string(index=False))
    print()
    print("Action: Keeping the first occurrence and removing duplicates.")
else:
    print("No duplicate registrations found in the member list.")

DUPLICATE REGISTRATIONS FOUND: 2 rows

  REG NO        MEMBERS
24/07453 Franklin Mrata
24/07453 Franklin Mrata

Action: Keeping the first occurrence and removing duplicates.


In [26]:
# 3c. Remove duplicate members (keep first occurrence)
members_clean = members_df.drop_duplicates(subset='REG NO', keep='first').copy()

print(f"Members after deduplication: {len(members_clean)}")
print(f"   Removed: {len(members_df) - len(members_clean)} duplicate(s)")

Members after deduplication: 40
   Removed: 1 duplicate(s)


In [27]:
# 3d. Validate registration number format
# Expected format: two digits, a forward slash, then 4-5 digits (e.g., 25/08756)

REG_PATTERN = r'^\d{2}/\d{4,5}$'

# Check members
members_clean['valid_reg'] = members_clean['REG NO'].apply(lambda x: bool(re.match(REG_PATTERN, x)))
invalid_member_regs = members_clean[~members_clean['valid_reg']]

if len(invalid_member_regs) > 0:
    print(f"{len(invalid_member_regs)} member(s) with invalid registration format:")
    print(invalid_member_regs[['REG NO', 'MEMBERS']].to_string(index=False))
else:
    print("All member registration numbers have valid format.")

print()

# Check ballots
ballots_df['valid_reg'] = ballots_df['REGISTRATION NUMBER'].apply(lambda x: bool(re.match(REG_PATTERN, x)))
invalid_ballot_regs = ballots_df[~ballots_df['valid_reg']]

if len(invalid_ballot_regs) > 0:
    print(f"{len(invalid_ballot_regs)} ballot(s) with invalid registration format:")
    print(invalid_ballot_regs[['REGISTRATION NUMBER']].to_string(index=False))
else:
    print("All ballot registration numbers have valid format.")

All member registration numbers have valid format.

All ballot registration numbers have valid format.


---

## Step 4: Check for Duplicate Votes

A critical integrity check: **Did anyone vote more than once?**

In a fair election, each registered member should submit exactly one ballot. If the same registration number appears multiple times in the ballot responses, we need to flag and handle it.

**Policy:** If duplicates are found, only the **first submission** is counted (based on timestamp).

In [28]:
# Check for duplicate ballot submissions
duplicate_votes = ballots_df[ballots_df.duplicated(subset='REGISTRATION NUMBER', keep=False)]

if len(duplicate_votes) > 0:
    print(f"DUPLICATE VOTES DETECTED: {len(duplicate_votes)} rows")
    print()
    print(duplicate_votes[['Timestamp', 'REGISTRATION NUMBER']].to_string(index=False))
    print()
    print("Action: Keeping only the first submission for each registration number.")
    ballots_clean = ballots_df.drop_duplicates(subset='REGISTRATION NUMBER', keep='first').copy()
else:
    print("No duplicate votes detected. Each registration number voted exactly once.")
    ballots_clean = ballots_df.copy()

print(f"\nValid ballots for counting: {len(ballots_clean)}")

No duplicate votes detected. Each registration number voted exactly once.

Valid ballots for counting: 39


---

## Step 5: Cross-Reference Voters Against Member List

We need to check:
- **How many voters are registered members?** These are valid internal votes.
- **How many voters are NOT in the member list?** These may be new or unregistered participants.
- **How many registered members did NOT vote?** This tells us about voter turnout.

### Important Note on Voter Privacy
We do NOT reveal how any individual voted. We only check whether they submitted a ballot (participated) or not.

In [29]:
# Create sets for comparison
member_reg_set = set(members_clean['REG NO'])
voter_reg_set = set(ballots_clean['REGISTRATION NUMBER'])

# Members who voted
members_who_voted = member_reg_set & voter_reg_set

# Members who did NOT vote
members_who_did_not_vote = member_reg_set - voter_reg_set

# Non-members who voted (voted but not in the official member list)
non_member_voters = voter_reg_set - member_reg_set

print("VOTER CROSS-REFERENCE RESULTS")
print(f"   Total registered members:         {len(member_reg_set)}")
print(f"   Total ballots cast:               {len(voter_reg_set)}")
print(f"   Registered members who voted:     {len(members_who_voted)}")
print(f"   Registered members who DID NOT vote: {len(members_who_did_not_vote)}")
print(f"   Non-member voters:                {len(non_member_voters)}")

# Turnout calculation
turnout = (len(members_who_voted) / len(member_reg_set)) * 100
print(f"\nVOTER TURNOUT: {turnout:.1f}%")

VOTER CROSS-REFERENCE RESULTS
   Total registered members:         40
   Total ballots cast:               39
   Registered members who voted:     24
   Registered members who DID NOT vote: 16
   Non-member voters:                15

VOTER TURNOUT: 60.0%


In [30]:
# Show members who did not vote (for follow-up, not to reveal their choices)
if members_who_did_not_vote:
    print(f"\nMembers who did not vote ({len(members_who_did_not_vote)}):")
    non_voters = members_clean[members_clean['REG NO'].isin(members_who_did_not_vote)]
    print(non_voters[['REG NO', 'MEMBERS']].to_string(index=False))

if non_member_voters:
    print(f"\nNon-member voters ({len(non_member_voters)}):")
    for reg in sorted(non_member_voters):
        print(f"   {reg}")


Members who did not vote (16):
  REG NO              MEMBERS
25/08062        Billy Lbaruni
24/06576 Churchil John Wesley
22/06872         David Wekesa
25/07987    Derrick Kipchumba
22/06569              Ezekiel
26/02432      Gikuyu Peterson
25/02332            Joy Ninah
22/00843      Judah Kiprotich
23/06525       Nyakio Wachira
25/02515     Odhiambo Derrick
26/04322        Radley chweya
24/07259       Roy Ian Nasser
24/07516       Samuel Kimathi
24/06646       Sharon Kimutai
25/08351           Thon Ayiik
25/00437         Venah Otieno

Non-member voters (15):
   20/02917
   22/07021
   22/09030
   24/04936
   24/07317
   25/02517
   25/03330
   25/05953
   25/07551
   25/08305
   25/08627
   25/08700
   25/09026
   25/09429
   26/03730


---

## Step 6: Identify Elective Positions and Candidates

The election covers the following positions. Let's identify the candidates running for each position based on the ballot responses.

A candidate is anyone whose name appears in the ballot column for a given position. An **uncontested** position is one where only a single candidate is listed.

In [31]:
# Define the position columns from the ballot
POSITIONS = [
    'PRESIDENT',
    'VICE PRESIDENT',
    'SECRETARY GENERAL',
    'ORGANISING SECRETARY',
    'PUBLICITY SECRETARY',
    'FINANCE SECRETARY'
]

print(f"Total Elective Positions: {len(POSITIONS)}")
print()

for position in POSITIONS:
    candidates = ballots_clean[position].unique()
    status = "UNCONTESTED" if len(candidates) == 1 else f"{len(candidates)} candidates"
    print(f"  {position}: {status}")
    for c in sorted(candidates):
        print(f"      • {c}")
    print()

Total Elective Positions: 6

  PRESIDENT: UNCONTESTED
      • Newton Ombaka

  VICE PRESIDENT: 2 candidates
      • Akankunda Shallon
      • Amilia Rehan

  SECRETARY GENERAL: UNCONTESTED
      • Osman Abdullahi

  ORGANISING SECRETARY: 3 candidates
      • Fiona Bett
      • George Mwai
      • Josephat Mulei

  PUBLICITY SECRETARY: UNCONTESTED
      • Nyamungu Achieng

  FINANCE SECRETARY: UNCONTESTED
      • Yolanda Nyambura Waceke



---

## Step 7: Count the Votes

This is the core of the election process. For each position, we count how many votes each candidate received.

**Method:** Simple majority — the candidate with the most votes wins.

For each position, we report:
- Total votes cast
- Each candidate's vote count and percentage
- The **winner**
- The **runner-up** (if the position was contested)
- **Margin of victory** (how many votes separated the winner from the runner-up)

In [32]:
# Count votes for each position
results = {}

for position in POSITIONS:
    vote_counts = ballots_clean[position].value_counts().sort_values(ascending=False)
    total_votes = vote_counts.sum()
    
    results[position] = {
        'counts': vote_counts,
        'total': total_votes,
        'winner': vote_counts.index[0],
        'winner_votes': vote_counts.iloc[0],
        'runner_up': vote_counts.index[1] if len(vote_counts) > 1 else None,
        'runner_up_votes': vote_counts.iloc[1] if len(vote_counts) > 1 else None,
        'is_uncontested': len(vote_counts) == 1
    }

print("Vote counting complete. Results below.\n")
print("=" * 70)

Vote counting complete. Results below.



In [ ]:
# Display vote totals as a consolidated results chart
vote_chart_rows = []
for position in POSITIONS:
    r = results[position]
    for candidate, votes in r['counts'].items():
        vote_chart_rows.append({
            'Position': position,
            'Candidate': candidate,
            'Votes': votes,
            'Vote share (%)': round(votes / r['total'] * 100, 1),
            'Result': 'Winner' if candidate == r['winner'] else 'Other candidate'
        })

vote_chart = pd.DataFrame(vote_chart_rows)
fig_vote_results = px.bar(
    vote_chart,
    x='Votes', y='Candidate', color='Result', text='Votes',
    facet_row='Position', orientation='h',
    category_orders={'Position': POSITIONS[::-1]},
    hover_data={'Vote share (%)': ':.1f'},
    color_discrete_map={'Winner': '#1F77B4', 'Other candidate': '#B8C2CC'},
    title='Election Results by Position'
)
fig_vote_results.update_layout(
    height=max(500, 230 * len(POSITIONS)),
    showlegend=True, legend_title_text='',
    margin=dict(t=80, l=30, r=30, b=40)
)
fig_vote_results.update_traces(textposition='outside', cliponaxis=False)
fig_vote_results.update_xaxes(title_text='Votes', showgrid=True, gridcolor='#E5E7EB')
fig_vote_results.update_yaxes(title_text='', categoryorder='total ascending')
fig_vote_results.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split('=')[-1]))

---

## Step 8: Election Summary Table

A consolidated summary of all results across all positions.

In [ ]:
# Create a summary table
summary_data = []
for position in POSITIONS:
    r = results[position]
    margin = r['winner_votes'] - r['runner_up_votes'] if r['runner_up_votes'] else r['winner_votes']
    summary_data.append({
        'Position': position,
        'Winner': r['winner'],
        'Winner Votes': r['winner_votes'],
        'Winner %': f"{(r['winner_votes'] / r['total']) * 100:.1f}%",
        'Runner-Up': r['runner_up'] or 'N/A (Unopposed)',
        'Runner-Up Votes': r['runner_up_votes'] if r['runner_up_votes'] else 'N/A',
        'Margin': margin,
        'Total Votes': r['total'],
        'Status': 'Contested' if not r['is_uncontested'] else 'Unopposed'
    })

summary_df = pd.DataFrame(summary_data)
print("\nELECTION RESULTS SUMMARY")
print("=" * 110)
print(summary_df.to_string(index=False))


ELECTION RESULTS SUMMARY
            Position                  Winner  Winner Votes Winner %       Runner-Up Runner-Up Votes  Margin  Total Votes    Status
           PRESIDENT           Newton Ombaka            39   100.0% N/A (Unopposed)             N/A      39           39 Unopposed
      VICE PRESIDENT       Akankunda Shallon            25    64.1%    Amilia Rehan              14      11           39 Contested
   SECRETARY GENERAL         Osman Abdullahi            39   100.0% N/A (Unopposed)             N/A      39           39 Unopposed
ORGANISING SECRETARY             George Mwai            21    53.8%      Fiona Bett              13       8           39 Contested
 PUBLICITY SECRETARY        Nyamungu Achieng            39   100.0% N/A (Unopposed)             N/A      39           39 Unopposed
   FINANCE SECRETARY Yolanda Nyambura Waceke            39   100.0% N/A (Unopposed)             N/A      39           39 Unopposed


---

## Step 9: Voter Participation by Year of Study

We can derive the **year of admission** (and therefore approximate year of study) from the registration number prefix. For example:
- `25/XXXXX` → Admitted in 2025 → Year 1
- `24/XXXXX` → Admitted in 2024 → Year 2
- `23/XXXXX` → Admitted in 2023 → Year 3

This allows us to analyze participation patterns by cohort.

In [ ]:
# Derive year of study from registration number
CURRENT_YEAR = 2026

def get_year_of_study(reg_no):
    """Derive year of study from the registration number prefix."""
    try:
        prefix = int(reg_no.split('/')[0])
        admission_year = 2000 + prefix if prefix <= 30 else 1900 + prefix
        year = CURRENT_YEAR - admission_year + 1
        return f'Year {year}' if 1 <= year <= 7 else 'Unknown'
    except:
        return 'Unknown'

# Add year of study to members
members_clean['Year of Study'] = members_clean['REG NO'].apply(get_year_of_study)

# Count registered members per year
year_registered = members_clean['Year of Study'].value_counts().sort_index()

# Count voters per year (only registered members who voted)
voted_members = members_clean[members_clean['REG NO'].isin(members_who_voted)]
year_voted = voted_members['Year of Study'].value_counts().sort_index()

# Combine into a table
year_stats = pd.DataFrame({
    'Registered': year_registered,
    'Voted': year_voted
}).fillna(0).astype(int)

year_stats['Turnout %'] = (year_stats['Voted'] / year_stats['Registered'] * 100).round(1)

print("VOTER TURNOUT BY YEAR OF STUDY")
print("=" * 50)
print(year_stats.to_string())

VOTER TURNOUT BY YEAR OF STUDY
               Registered  Voted  Turnout %
Year of Study                              
Year 1                  9      7       77.8
Year 2                 14      8       57.1
Year 3                  6      2       33.3
Year 4                  3      2       66.7
Year 5                  7      4       57.1
Year 6                  1      1      100.0


---

## Step 10: Visual Analysis

The charts below summarize participation and election outcomes without exposing individual ballot choices.

In [ ]:
# Participation overview
participation_df = pd.DataFrame({
    'Group': ['Registered members who voted', 'Registered members who did not vote', 'Non-member voters'],
    'Count': [len(members_who_voted), len(members_who_did_not_vote), len(non_member_voters)]
})

fig_participation = px.bar(
    participation_df, x='Group', y='Count', text='Count',
    title='Voter Participation Overview',
    color='Group', color_discrete_sequence=px.colors.qualitative.Safe
)
fig_participation.update_layout(showlegend=False, xaxis_title='', yaxis_title='Number of people')
fig_participation.update_traces(textposition='outside')
fig_participation.show()

In [ ]:
# Winning share by position
result_chart = summary_df.copy()
result_chart['Winning share'] = result_chart['Winner %'].str.rstrip('%').astype(float)

fig_results = px.bar(
    result_chart, x='Position', y='Winning share', color='Status', text='Winner %',
    hover_data=['Winner', 'Winner Votes', 'Margin'],
    title='Winning Share by Position',
    color_discrete_map={'Contested': '#2A9D8F', 'Unopposed': '#E9C46A'}
)
fig_results.update_layout(xaxis_title='', yaxis_title='Winning share (%)', yaxis_range=[0, 105])
fig_results.update_traces(textposition='outside')
fig_results.show()

---

## Step 11: Data Quality Summary

A final summary of all data quality checks performed during this audit.

In [ ]:
print("\nDATA QUALITY AUDIT REPORT")
print("=" * 60)
print()

checks = [
    ('Duplicate Member Registrations', len(duplicate_members) // 2, 
     'PASS' if len(duplicate_members) == 0 else 'REVIEW'),
    ('Invalid Member Reg Numbers', len(invalid_member_regs),
     'PASS' if len(invalid_member_regs) == 0 else 'REVIEW'),
    ('Duplicate Votes', len(duplicate_votes) // 2,
     'PASS' if len(duplicate_votes) == 0 else 'REVIEW'),
    ('Invalid Ballot Reg Numbers', len(invalid_ballot_regs),
     'PASS' if len(invalid_ballot_regs) == 0 else 'REVIEW'),
    ('Non-Member Voters', len(non_member_voters),
     'PASS' if len(non_member_voters) == 0 else 'REVIEW'),
    ('Members Who Did Not Vote', len(members_who_did_not_vote),
     'INFO')
]

for check_name, count, status in checks:
    print(f"   {status}  {check_name}: {count}")

print()
print("=" * 60)
print(f"   Total Registered Members (after dedup): {len(members_clean)}")
print(f"   Total Ballots Cast: {len(ballots_clean)}")
print(f"   Total Valid Votes: {len(ballots_clean[ballots_clean['valid_reg']])}")
print(f"   Total Invalid Votes: {len(ballots_clean[~ballots_clean['valid_reg']])}")
print(f"   Voter Turnout (registered members): {turnout:.1f}%")


DATA QUALITY AUDIT REPORT

   REVIEW  Duplicate Member Registrations: 1
   PASS  Invalid Member Reg Numbers: 0
   PASS  Duplicate Votes: 0
   PASS  Invalid Ballot Reg Numbers: 0
   REVIEW  Non-Member Voters: 15
   INFO  Members Who Did Not Vote: 16

   Total Registered Members (after dedup): 40
   Total Ballots Cast: 39
   Total Valid Votes: 39
   Total Invalid Votes: 0
   Voter Turnout (registered members): 60.0%


---

## Step 12: Official Elected Officials

Based on the vote counts above, the following candidates have been declared winners of the KCAU Debate Club Elections 2026/27:

In [ ]:
print("\n" + "=" * 60)
print(" OFFICIAL ELECTION RESULTS — KCAU DEBATE CLUB 2026/27")
print("=" * 60)
print()

for position in POSITIONS:
    r = results[position]
    status = '(Unopposed)' if r['is_uncontested'] else f'({r["winner_votes"]} votes, {(r["winner_votes"]/r["total"]*100):.1f}%)'
    print(f"    {position}: {r['winner']} {status}")

print()
print("=" * 60)
print(f"    Election Date: July 23, 2026")
print(f"    Voter Turnout: {turnout:.1f}% (registered members)")
print(f"    Total Ballots: {len(ballots_clean)}")
print("=" * 60)


 OFFICIAL ELECTION RESULTS — KCAU DEBATE CLUB 2026/27

    PRESIDENT: Newton Ombaka (Unopposed)
    VICE PRESIDENT: Akankunda Shallon (25 votes, 64.1%)
    SECRETARY GENERAL: Osman Abdullahi (Unopposed)
    ORGANISING SECRETARY: George Mwai (21 votes, 53.8%)
    PUBLICITY SECRETARY: Nyamungu Achieng (Unopposed)
    FINANCE SECRETARY: Yolanda Nyambura Waceke (Unopposed)

    Election Date: July 23, 2026
    Voter Turnout: 60.0% (registered members)
    Total Ballots: 39


---

## Audit Notes

1. **Data Integrity:** All ballot data was sourced directly from Google Forms responses. No manual modifications were made to the raw data.

2. **Duplicate Handling:** Duplicate member registrations were resolved by keeping the first occurrence. Duplicate ballot submissions (if any) would keep only the first submission by timestamp.

3. **Privacy:** This audit does not reveal how any individual voted. Only aggregate counts and participation status (voted/did not vote) are reported.

4. **Non-Member Voters:** Ballots from registration numbers not found in the official member list are counted in the overall totals but are flagged for review. These may represent new members not yet added to the register.

---

*This notebook was prepared as part of the official election audit process for the KCAU Debate Club Elections 2026/27.*